In [2]:
# Run this once, then comment it out
#!pip install faker tqdm --quiet


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: C:\Users\abhir\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [4]:
!pip install faker pandas numpy scipy tqdm --quie
# =============================================================================
# DIGITAL LENDING: PORTFOLIO OPTIMIZATION
# MODULE 1: SYNTHETIC DATA GENERATION
# =============================================================================
# Author   : Senior Fintech Data Scientist
# Purpose  : Generate production-style synthetic datasets for a digital lending
#            ecosystem covering credit risk, portfolio optimization, delinquency
#            prediction, customer segmentation, pricing & early-warning systems.
# Tested   : Google Colab & Jupyter Notebook
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Install dependencies (run once; safe to re-run)
# ─────────────────────────────────────────────────────────────────────────────
# Uncomment the line below when running in Google Colab
# !pip install faker pandas numpy scipy tqdm --quiet

# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports & global configuration
# ─────────────────────────────────────────────────────────────────────────────
import os
import logging
import warnings
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker
from scipy.stats import beta as beta_dist
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("en_IN")          # Indian locale for realistic names/cities
Faker.seed(SEED)

# ── Volume knobs (scale up/down here only) ───────────────────────────────────
N_CUSTOMERS = 25_000
N_LOANS     = 65_000           # ≥ 60 000 as required

# ── Output folder ────────────────────────────────────────────────────────────
OUTPUT_DIR = "lending_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

log.info("Configuration loaded. N_CUSTOMERS=%d  N_LOANS=%d", N_CUSTOMERS, N_LOANS)


# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Shared lookup tables (reused across all generators)
# ─────────────────────────────────────────────────────────────────────────────

# Indian cities mapped to state and urban/semi-urban flag
CITY_MAP = {
    # city: (state, urban_flag)
    "Mumbai":      ("Maharashtra",    "Urban"),
    "Delhi":       ("Delhi",          "Urban"),
    "Bengaluru":   ("Karnataka",      "Urban"),
    "Hyderabad":   ("Telangana",      "Urban"),
    "Chennai":     ("Tamil Nadu",     "Urban"),
    "Kolkata":     ("West Bengal",    "Urban"),
    "Pune":        ("Maharashtra",    "Urban"),
    "Ahmedabad":   ("Gujarat",        "Urban"),
    "Jaipur":      ("Rajasthan",      "Semi-Urban"),
    "Lucknow":     ("Uttar Pradesh",  "Semi-Urban"),
    "Nagpur":      ("Maharashtra",    "Semi-Urban"),
    "Indore":      ("Madhya Pradesh", "Semi-Urban"),
    "Bhopal":      ("Madhya Pradesh", "Semi-Urban"),
    "Patna":       ("Bihar",          "Semi-Urban"),
    "Vadodara":    ("Gujarat",        "Semi-Urban"),
    "Coimbatore":  ("Tamil Nadu",     "Semi-Urban"),
    "Visakhapatnam":("Andhra Pradesh","Semi-Urban"),
    "Agra":        ("Uttar Pradesh",  "Semi-Urban"),
    "Kanpur":      ("Uttar Pradesh",  "Semi-Urban"),
    "Nashik":      ("Maharashtra",    "Semi-Urban"),
}
CITIES        = list(CITY_MAP.keys())
CITY_WEIGHTS  = (
    [0.07]*8 +   # 8 urban cities — slightly higher weight
    [0.03]*12    # 12 semi-urban cities
)
# Normalise weights
CITY_WEIGHTS  = np.array(CITY_WEIGHTS) / sum(CITY_WEIGHTS)

EMPLOYMENT_TYPES = ["Salaried", "Self-Employed", "Gig Worker",
                    "First-Time Borrower", "SME Owner"]
EDUCATION_LEVELS = ["Below Matriculation", "Matriculation",
                    "Higher Secondary", "Graduate", "Post-Graduate"]
MARITAL_STATUSES = ["Single", "Married", "Divorced", "Widowed"]
DEVICE_TYPES     = ["Android", "iOS", "Feature Phone", "Desktop"]
ACQ_CHANNELS     = ["Organic Search", "Referral", "Social Media",
                    "DSA Partner", "Bank Branch", "App Store", "Email Campaign"]
LOAN_TYPES       = ["Personal Loan", "SME Working Capital", "BNPL"]

# Macroeconomic stress windows (origination months that see +30% delinquency)
STRESS_MONTHS    = pd.date_range("2022-04-01", periods=4, freq="MS").tolist() + \
                   pd.date_range("2023-10-01", periods=3, freq="MS").tolist()


# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Helper / utility functions
# ─────────────────────────────────────────────────────────────────────────────

def clamp(value, lo, hi):
    """Clamp a value to [lo, hi]."""
    return max(lo, min(hi, value))


def add_noise(series: pd.Series, pct: float = 0.05) -> pd.Series:
    """Add Gaussian noise (pct * std) to a numeric Series."""
    noise = np.random.normal(0, pct * series.std(), len(series))
    return series + noise


def introduce_missing(df: pd.DataFrame, col: str, rate: float = 0.03) -> pd.DataFrame:
    """Randomly null out `rate` fraction of a column (realistic missingness)."""
    mask = np.random.random(len(df)) < rate
    df.loc[mask, col] = np.nan
    return df


def introduce_outliers(series: pd.Series, pct: float = 0.005) -> pd.Series:
    """Replace a tiny fraction of values with extreme outliers."""
    n = int(len(series) * pct)
    if n == 0:
        return series
        
    idx = np.random.choice(series.index, n, replace=False)
    
    # 1. Compute the outlier values as a temporary array
    outliers = series.mean() + series.std() * np.random.choice([-5, 5], n)
    
    # 2. Safely cast the outliers to match the Series' native data type (int)
    outliers = outliers.astype(series.dtype)
    
    # 3. Assign them back without triggering a LossySetitemError
    series.loc[idx] = outliers
    return series


def sigmoid(x):
    """Sigmoid activation — maps any real to (0,1)."""
    return 1 / (1 + np.exp(-x))


# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — TABLE 1: CUSTOMER PROFILE
# ─────────────────────────────────────────────────────────────────────────────

def generate_customer_profiles(n: int = N_CUSTOMERS) -> pd.DataFrame:
    """
    Build the master customer table with realistic demographic,
    financial and digital attributes.

    Key business rules encoded:
    • Income depends on employment type (SME > Salaried > Gig > First-Time)
    • Credit score correlates with income stability + credit history
    • Underbanked / thin-file customers have shorter histories
    • Semi-urban customers show lower incomes and credit scores on average
    • Digital engagement drives channel attribution
    """
    log.info("Generating %d customer profiles …", n)

    # ── Customer IDs ─────────────────────────────────────────────────────────
    customer_ids = [f"CUST{str(i).zfill(7)}" for i in range(1, n + 1)]

    # ── Demographics ─────────────────────────────────────────────────────────
    ages    = np.random.randint(21, 65, n)
    genders = np.random.choice(["Male", "Female", "Other"],
                               n, p=[0.58, 0.40, 0.02])

    cities  = np.random.choice(CITIES, n, p=CITY_WEIGHTS)
    states  = [CITY_MAP[c][0] for c in cities]
    urban   = [CITY_MAP[c][1] for c in cities]
    is_urban = np.array([1 if u == "Urban" else 0 for u in urban])

    emp_types = np.random.choice(
        EMPLOYMENT_TYPES, n,
        p=[0.40, 0.22, 0.15, 0.13, 0.10]   # Salaried most common
    )
    education = np.random.choice(
        EDUCATION_LEVELS, n,
        p=[0.05, 0.12, 0.23, 0.42, 0.18]
    )
    marital   = np.random.choice(
        MARITAL_STATUSES, n,
        p=[0.32, 0.55, 0.08, 0.05]
    )
    dependents = np.random.choice([0, 1, 2, 3, 4], n, p=[0.20, 0.25, 0.30, 0.18, 0.07])

    # ── Income (employment-type aware) ───────────────────────────────────────
    income_params = {
        "Salaried":           (45_000, 18_000),
        "Self-Employed":      (52_000, 28_000),
        "Gig Worker":         (22_000, 10_000),
        "First-Time Borrower":(18_000,  8_000),
        "SME Owner":          (85_000, 45_000),
    }
    monthly_income = np.array([
        clamp(np.random.normal(*income_params[e]), 5_000, 500_000)
        for e in emp_types
    ])
    # Semi-urban customers earn ~20% less on average
    monthly_income = monthly_income * (0.80 + 0.20 * is_urban)
    monthly_income = add_noise(pd.Series(monthly_income)).values

    # ── Income stability score (0–1) ─────────────────────────────────────────
    # Salaried = high stability; Gig = low stability
    base_stability = {
        "Salaried": 0.80, "Self-Employed": 0.60,
        "Gig Worker": 0.40, "First-Time Borrower": 0.55, "SME Owner": 0.65,
    }
    income_stability = np.array([
        clamp(np.random.normal(base_stability[e], 0.12), 0.0, 1.0)
        for e in emp_types
    ])

    # ── Credit history length (months) ───────────────────────────────────────
    # Thin-file: first-time borrowers and younger customers
    age_factor = (ages - 21) / 44   # normalised 0–1
    hist_base  = age_factor * 84    # max ~7 years
    # First-time borrowers and gig workers: thin file
    thin_file_mask = np.isin(emp_types, ["First-Time Borrower", "Gig Worker"])
    hist_base[thin_file_mask] *= 0.3
    credit_history = np.clip(
        hist_base + np.random.normal(0, 8, n), 0, 120
    ).astype(int)

    # ── Credit score (300–900) ────────────────────────────────────────────────
    # Driven by: income_stability + history length + education + income level
    norm_income  = np.clip((monthly_income - 5_000) / 495_000, 0, 1)
    norm_history = credit_history / 120
    edu_score    = np.array(
        [EDUCATION_LEVELS.index(e) / 4 for e in education]
    )
    raw_credit = (
        0.35 * income_stability
        + 0.25 * norm_history
        + 0.20 * norm_income
        + 0.10 * edu_score
        + 0.10 * np.random.random(n)
    )
    credit_score = (300 + raw_credit * 600).astype(int)
    credit_score = introduce_outliers(pd.Series(credit_score)).values.astype(int)
    credit_score = np.clip(credit_score, 300, 900)

    # ── Bank balance average ──────────────────────────────────────────────────
    bank_balance_avg = monthly_income * np.random.uniform(0.5, 3.5, n)
    bank_balance_avg = add_noise(pd.Series(bank_balance_avg)).values

    # ── Existing loans count ──────────────────────────────────────────────────
    existing_loans = np.random.choice([0, 1, 2, 3, 4], n,
                                      p=[0.35, 0.30, 0.20, 0.10, 0.05])
    # First-time borrowers: no prior loans
    existing_loans[np.array(emp_types) == "First-Time Borrower"] = 0

    # ── Digital engagement score (0–100) ─────────────────────────────────────
    # Younger, urban, Android users → higher engagement
    urban_boost  = is_urban * 10
    age_penalty  = np.clip((ages - 21) / 44 * -20, -20, 0)
    digital_eng  = np.clip(
        50 + urban_boost + age_penalty + np.random.normal(0, 12, n),
        0, 100
    ).astype(int)

    # ── Device type ───────────────────────────────────────────────────────────
    device_type = np.random.choice(
        DEVICE_TYPES, n, p=[0.60, 0.22, 0.12, 0.06]
    )

    # ── Acquisition channel ───────────────────────────────────────────────────
    # High digital engagement → Organic Search / App Store
    acq_channel = []
    for eng in digital_eng:
        if eng > 70:
            acq_channel.append(np.random.choice(
                ["Organic Search", "App Store", "Social Media"], p=[0.45, 0.35, 0.20]
            ))
        elif eng > 40:
            acq_channel.append(np.random.choice(
                ["Referral", "Social Media", "DSA Partner", "Email Campaign"],
                p=[0.30, 0.25, 0.25, 0.20]
            ))
        else:
            acq_channel.append(np.random.choice(
                ["Bank Branch", "DSA Partner", "Referral"], p=[0.45, 0.35, 0.20]
            ))

    # ── Build DataFrame ───────────────────────────────────────────────────────
    df = pd.DataFrame({
        "customer_id":            customer_ids,
        "age":                    ages,
        "gender":                 genders,
        "city":                   cities,
        "state":                  states,
        "urban_semiurban_flag":   urban,
        "employment_type":        emp_types,
        "monthly_income":         monthly_income.round(2),
        "income_stability_score": income_stability.round(4),
        "education_level":        education,
        "marital_status":         marital,
        "dependents":             dependents,
        "bank_balance_avg":       bank_balance_avg.round(2),
        "existing_loans":         existing_loans,
        "credit_score":           credit_score,
        "credit_history_length":  credit_history,
        "device_type":            device_type,
        "digital_engagement_score": digital_eng,
        "acquisition_channel":    acq_channel,
    })

    # ── Introduce realistic imperfections ────────────────────────────────────
    for col, rate in [("bank_balance_avg", 0.04),
                      ("credit_history_length", 0.02),
                      ("income_stability_score", 0.03)]:
        df = introduce_missing(df, col, rate)

    log.info("Customer profiles done. Shape: %s", df.shape)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — TABLE 2: LOAN APPLICATION
# ─────────────────────────────────────────────────────────────────────────────

def generate_loan_applications(customers: pd.DataFrame,
                                n_loans: int = N_LOANS) -> pd.DataFrame:
    """
    Simulate loan originations.

    Key business rules:
    • Loan type mix: Personal 55%, SME 25%, BNPL 20%
    • BNPL → small ticket (₹2K–₹25K), short tenure
    • SME  → large ticket (₹1L–₹20L), long tenure
    • Interest rate increases with risk grade
    • Approval probability decreases with credit risk
    • Processing time faster for digital, high-engagement customers
    """
    log.info("Generating %d loan applications …", n_loans)

    # ── Sample customers (with replacement — repeat borrowers allowed) ────────
    sampled = customers.sample(n_loans, replace=True, random_state=SEED).reset_index(drop=True)

    loan_ids = [f"LOAN{str(i).zfill(8)}" for i in range(1, n_loans + 1)]

    # ── Loan type ─────────────────────────────────────────────────────────────
    loan_type = np.random.choice(LOAN_TYPES, n_loans, p=[0.55, 0.25, 0.20])

    # ── Loan amount by type ───────────────────────────────────────────────────
    loan_amounts = []
    for lt, inc in zip(loan_type, sampled["monthly_income"]):
        if lt == "BNPL":
            amt = np.random.uniform(2_000, 25_000)
        elif lt == "SME Working Capital":
            amt = np.random.uniform(100_000, 2_000_000)
        else:  # Personal
            amt = np.random.uniform(10_000, 500_000)
        # Cap at ~40× monthly income (sanity check)
        amt = min(amt, inc * 40)
        loan_amounts.append(round(amt, 2))
    loan_amounts = np.array(loan_amounts)

    # ── Tenure (months) ───────────────────────────────────────────────────────
    tenure_map = {
        "BNPL": (1, 6),
        "Personal Loan": (6, 60),
        "SME Working Capital": (12, 84),
    }
    loan_tenure = np.array([
        np.random.randint(*tenure_map[lt]) for lt in loan_type
    ])

    # ── Risk grade (A–E) based on credit score ─────────────────────────────
    def assign_grade(cs):
        if cs >= 750: return "A"
        elif cs >= 700: return "B"
        elif cs >= 650: return "C"
        elif cs >= 600: return "D"
        else: return "E"

    credit_scores = sampled["credit_score"].fillna(500).astype(int)
    risk_grade = credit_scores.apply(assign_grade).values

    # ── Interest rate (risk-adjusted) ─────────────────────────────────────────
    grade_rate = {"A": 10, "B": 14, "C": 18, "D": 22, "E": 28}
    base_rate  = np.array([grade_rate[g] for g in risk_grade], dtype=float)
    # Add noise and BNPL premium
    bnpl_premium = np.where(loan_type == "BNPL", 3, 0)
    interest_rate = np.clip(
        base_rate + bnpl_premium + np.random.normal(0, 1.5, n_loans),
        8, 36
    ).round(2)

    # ── Origination date (Jan 2020 – Dec 2024) ────────────────────────────────
    start = datetime(2020, 1, 1)
    end   = datetime(2024, 12, 31)
    delta_days = (end - start).days
    orig_dates = [
        start + timedelta(days=int(d))
        for d in np.random.randint(0, delta_days, n_loans)
    ]

    # ── Approval status ───────────────────────────────────────────────────────
    # Probability of approval:
    #   base = 0.80, reduced by risk grade, existing loans, large amounts
    grade_penalty = {"A": 0, "B": -0.05, "C": -0.15, "D": -0.25, "E": -0.40}
    amount_norm   = np.clip(loan_amounts / 2_000_000, 0, 1)
    existing_pen  = sampled["existing_loans"].fillna(0).values * 0.05

    p_approve = np.clip(
        0.80
        + np.array([grade_penalty[g] for g in risk_grade])
        - 0.10 * amount_norm
        - existing_pen
        + np.random.normal(0, 0.03, n_loans),
        0.05, 0.98
    )
    approval_status = np.where(
        np.random.random(n_loans) < p_approve, "Approved", "Rejected"
    )

    # ── Processing time (hours) ───────────────────────────────────────────────
    # High digital engagement → faster; risky borrowers → slower
    eng_factor = sampled["digital_engagement_score"].fillna(50).values / 100
    grade_time = {"A": 2, "B": 4, "C": 8, "D": 12, "E": 20}
    proc_time  = np.clip(
        np.array([grade_time[g] for g in risk_grade])
        - 5 * eng_factor
        + np.random.exponential(2, n_loans),
        0.5, 72
    ).round(1)

    # ── Acquisition cost (₹) ─────────────────────────────────────────────────
    channel_cost = {
        "Organic Search": 250, "App Store": 300, "Social Media": 500,
        "Referral": 400, "DSA Partner": 1200, "Bank Branch": 800,
        "Email Campaign": 350,
    }
    acq_cost = np.array([
        channel_cost.get(ch, 500) * np.random.uniform(0.8, 1.4)
        for ch in sampled["acquisition_channel"]
    ]).round(2)

    # ── EMI amount ────────────────────────────────────────────────────────────
    def calc_emi(principal, annual_rate, months):
        if months <= 0: return principal
        r = annual_rate / 12 / 100
        if r == 0: return principal / months
        return principal * r * (1 + r)**months / ((1 + r)**months - 1)

    emi = np.array([
        round(calc_emi(la, ir, tm), 2)
        for la, ir, tm in zip(loan_amounts, interest_rate, loan_tenure)
    ])

    # ── Collateral flag (mostly SME) ──────────────────────────────────────────
    collateral = np.where(
        loan_type == "SME Working Capital",
        np.random.choice([0, 1], n_loans, p=[0.35, 0.65]),
        np.random.choice([0, 1], n_loans, p=[0.90, 0.10]),
    )

    # ── Repayment frequency ───────────────────────────────────────────────────
    repay_freq = np.where(
        loan_type == "BNPL", "Weekly",
        np.random.choice(["Monthly", "Bi-Monthly"], n_loans, p=[0.85, 0.15])
    )

    df = pd.DataFrame({
        "loan_id":               loan_ids,
        "customer_id":           sampled["customer_id"].values,
        "loan_type":             loan_type,
        "loan_amount":           loan_amounts,
        "loan_tenure_months":    loan_tenure,
        "interest_rate":         interest_rate,
        "origination_date":      orig_dates,
        "approval_status":       approval_status,
        "origination_risk_grade":risk_grade,
        "processing_time_hours": proc_time,
        "acquisition_cost":      acq_cost,
        "collateral_flag":       collateral,
        "repayment_frequency":   repay_freq,
        "emi_amount":            emi,
    })

    # ── Missing values ────────────────────────────────────────────────────────
    df = introduce_missing(df, "processing_time_hours", 0.02)
    df = introduce_missing(df, "acquisition_cost", 0.03)

    log.info("Loan applications done. Shape: %s | Approval rate: %.1f%%",
             df.shape, (df["approval_status"] == "Approved").mean() * 100)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — TABLE 3: REPAYMENT BEHAVIOR
# ─────────────────────────────────────────────────────────────────────────────

def generate_repayment_behavior(loans: pd.DataFrame,
                                 customers: pd.DataFrame) -> pd.DataFrame:
    """
    Generate monthly repayment records for APPROVED loans.

    Key business rules:
    • Repayment depends on income stability + credit score + behavioural volatility
    • Macro-stress months add +30% delinquency probability
    • Delinquency follows a realistic progression:
        Current → DPD_30 → DPD_60 → DPD_90 → Write-Off
    • Once in write-off, no recovery in current record
    • Overall default rate targets 8–18%
    """
    log.info("Generating repayment records (this may take ~30–60 s) …")

    approved = loans[loans["approval_status"] == "Approved"].copy()
    cust_map  = customers.set_index("customer_id")[
        ["income_stability_score", "credit_score"]
    ].to_dict("index")

    records = []
    repay_id = 1

    for _, loan in tqdm(approved.iterrows(), total=len(approved), desc="Repayments"):
        cid  = loan["customer_id"]
        info = cust_map.get(cid, {})
        stab = info.get("income_stability_score", 0.5) or 0.5
        cs   = info.get("credit_score", 600) or 600

        # Base probability of a bad payment for this borrower
        grade_default = {"A": 0.04, "B": 0.08, "C": 0.14, "D": 0.22, "E": 0.32}
        p_bad = grade_default[loan["origination_risk_grade"]]
        p_bad = p_bad * (1.5 - stab)  # high stability reduces default probability

        tenure      = int(loan["loan_tenure_months"])
        orig_date   = pd.Timestamp(loan["origination_date"])
        delinquency = "Current"      # progressive state
        bounce_cnt  = 0

        for month_idx in range(tenure):
            due_date = orig_date + pd.DateOffset(months=month_idx + 1)

            # Macro-stress adjustment
            stress_mult = 1.30 if due_date.to_period("M").to_timestamp() in [
                s.to_period("M").to_timestamp() for s in STRESS_MONTHS
            ] else 1.0

            # Worsening trend for risky borrowers over time
            time_decay = 1 + 0.008 * month_idx   # risk slowly accumulates

            effective_p = clamp(p_bad * stress_mult * time_decay, 0.01, 0.95)
            rand_draw   = np.random.random()

            if rand_draw < effective_p * 0.08:     # worst — missed payment
                payment_date   = None
                payment_status = "Missed"
                delay_days     = None
                partial        = 0
                missed         = 1
                bounce_cnt    += 1
            elif rand_draw < effective_p * 0.25:   # partial payment
                pay_delay      = np.random.randint(1, 45)
                payment_date   = due_date + timedelta(days=int(pay_delay))
                payment_status = "Partial"
                delay_days     = pay_delay
                partial        = 1
                missed         = 0
            elif rand_draw < effective_p:           # late payment
                pay_delay      = np.random.randint(1, 30)
                payment_date   = due_date + timedelta(days=int(pay_delay))
                payment_status = "Late"
                delay_days     = pay_delay
                partial        = 0
                missed         = 0
            else:                                   # on-time
                pay_delay      = np.random.randint(-3, 3)
                payment_date   = due_date + timedelta(days=int(pay_delay))
                payment_status = "On-Time"
                delay_days     = max(0, pay_delay)
                partial        = 0
                missed         = 0

            # ── Delinquency stage progression ────────────────────────────────
            if payment_status == "Missed":
                if delinquency == "Current":
                    delinquency = "DPD_30"
                elif delinquency == "DPD_30":
                    delinquency = "DPD_60"
                elif delinquency == "DPD_60":
                    delinquency = "DPD_90"
                elif delinquency == "DPD_90":
                    delinquency = "Write-Off"
            elif payment_status == "On-Time":
                # Partial recovery for good payers
                if delinquency == "DPD_30":
                    delinquency = "Current"

            records.append({
                "repayment_id":      f"REP{str(repay_id).zfill(10)}",
                "loan_id":           loan["loan_id"],
                "due_date":          due_date.date(),
                "payment_date":      payment_date.date() if payment_date else None,
                "payment_status":    payment_status,
                "delay_days":        delay_days,
                "partial_payment_flag": partial,
                "missed_payment_flag":  missed,
                "bounce_count":      bounce_cnt,
                "delinquency_stage": delinquency,
            })
            repay_id += 1

            if delinquency == "Write-Off":
                break   # no more records after write-off

    df = pd.DataFrame(records)
    log.info("Repayment table done. Shape: %s", df.shape)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — TABLE 4: BEHAVIORAL SIGNALS
# ─────────────────────────────────────────────────────────────────────────────

def generate_behavioral_signals(customers: pd.DataFrame,
                                  loans: pd.DataFrame) -> pd.DataFrame:
    """
    Monthly behavioral / transaction signals per customer.

    Key business rules:
    • High volatility → higher delinquency risk
    • Financially stressed customers show deteriorating signals over time
    • Covers the combined origination window of the loan portfolio
    """
    log.info("Generating behavioral signals …")

    months = pd.date_range("2020-01-01", "2025-01-01", freq="MS")
    active_customers = loans["customer_id"].unique()
    cust_sub = customers[customers["customer_id"].isin(active_customers)].copy()

    records = []
    sig_id  = 1

    for _, cust in tqdm(cust_sub.iterrows(), total=len(cust_sub), desc="BehSignals"):
        stab    = cust["income_stability_score"] if pd.notna(cust["income_stability_score"]) else 0.5
        cs      = cust["credit_score"] if pd.notna(cust["credit_score"]) else 600

        # Base volatility — low stability customers are more volatile
        base_vol = clamp(1.0 - stab, 0.1, 0.9)

        for month in months:
            # Trend: volatility drifts up slightly each quarter for risky customers
            quarter_idx = (month.year - 2020) * 4 + month.quarter
            drift       = 0.005 * quarter_idx if base_vol > 0.5 else 0

            spend_vol    = clamp(
                base_vol + drift + np.random.normal(0, 0.08), 0, 1
            )
            bal_vol      = clamp(
                base_vol * 0.9 + drift + np.random.normal(0, 0.07), 0, 1
            )
            txn_freq     = max(1, int(np.random.normal(
                20 * (1 - base_vol * 0.5), 5
            )))
            shock_flag   = 1 if (spend_vol > 0.75 and np.random.random() < 0.25) else 0
            cashflow_con = clamp(
                stab * 0.8 + (cs / 900) * 0.2 + np.random.normal(0, 0.07), 0, 1
            )
            app_usage    = max(0, int(np.random.normal(
                cust["digital_engagement_score"] * 0.3, 8
            )))
            loc_change   = np.random.poisson(2 if base_vol > 0.6 else 0.5)
            late_night   = clamp(
                0.05 + base_vol * 0.15 + np.random.normal(0, 0.03), 0, 0.5
            )

            records.append({
                "signal_id":                f"SIG{str(sig_id).zfill(10)}",
                "customer_id":              cust["customer_id"],
                "month":                    month.date(),
                "spending_volatility":      round(spend_vol, 4),
                "balance_volatility":       round(bal_vol, 4),
                "transaction_frequency":    txn_freq,
                "spending_shock_flag":      shock_flag,
                "cashflow_consistency_score": round(cashflow_con, 4),
                "app_usage_frequency":      app_usage,
                "location_change_frequency":loc_change,
                "late_night_transaction_ratio": round(late_night, 4),
            })
            sig_id += 1

    df = pd.DataFrame(records)
    # Introduce some missing app_usage (offline customers)
    df = introduce_missing(df, "app_usage_frequency", 0.04)
    log.info("Behavioral signals done. Shape: %s", df.shape)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — TABLE 5: OUTCOME TABLE
# ─────────────────────────────────────────────────────────────────────────────

def generate_outcomes(loans: pd.DataFrame,
                       repayments: pd.DataFrame,
                       customers: pd.DataFrame) -> pd.DataFrame:
    """
    Derive loan-level outcome metrics from repayment behavior.

    Key business rules:
    • Default = any loan that reaches DPD_90 or Write-Off stage
    • Profitability = interest earned – acquisition cost – expected loss
    • CLV depends on repayment quality, income, and product mix
    • Recovery amount estimated for defaulted loans (10–40% of balance)
    """
    log.info("Generating outcome table …")

    approved = loans[loans["approval_status"] == "Approved"].copy()
    cust_map  = customers.set_index("customer_id")[
        ["monthly_income", "acquisition_channel"]
    ].to_dict("index")

    # ── Worst delinquency stage per loan ─────────────────────────────────────
    stage_order = {"Current": 0, "DPD_30": 1, "DPD_60": 2,
                   "DPD_90": 3, "Write-Off": 4}
    repayments["stage_rank"] = repayments["delinquency_stage"].map(stage_order)
    worst_stage = (
        repayments.groupby("loan_id")["stage_rank"].max()
        .reset_index()
        .rename(columns={"stage_rank": "worst_stage_rank"})
    )
    stage_inv   = {v: k for k, v in stage_order.items()}
    worst_stage["worst_stage"] = worst_stage["worst_stage_rank"].map(stage_inv)

    approved = approved.merge(worst_stage, on="loan_id", how="left")
    approved["worst_stage"].fillna("Current", inplace=True)
    approved["worst_stage_rank"].fillna(0, inplace=True)

    # ── Default & write-off flags ─────────────────────────────────────────────
    default_flag  = (approved["worst_stage_rank"] >= 3).astype(int)
    writeoff_flag = (approved["worst_stage"] == "Write-Off").astype(int)

    # ── Recovery amount (for defaults) ───────────────────────────────────────
    recovery_pct    = np.where(
        writeoff_flag == 1,
        np.random.uniform(0.10, 0.40, len(approved)),
        np.where(default_flag == 1, np.random.uniform(0.40, 0.75, len(approved)), 1.0)
    )
    recovery_amount = (approved["loan_amount"].values * recovery_pct).round(2)

    # ── Total interest earned ─────────────────────────────────────────────────
    tenure_yrs    = approved["loan_tenure_months"].values / 12
    interest_earned = (
        approved["loan_amount"].values
        * approved["interest_rate"].values / 100
        * tenure_yrs
        * (1 - default_flag.values * 0.5)    # defaults earn less interest
    ).round(2)

    # ── Profitability score ───────────────────────────────────────────────────
    # profit = interest_earned - acquisition_cost - expected_loss
    expected_loss = approved["loan_amount"].values * default_flag.values * (1 - recovery_pct)
    acq_cost      = approved["acquisition_cost"].fillna(500).values
    raw_profit    = interest_earned - acq_cost - expected_loss
    # Normalise to -1 → 1
    max_abs = np.percentile(np.abs(raw_profit), 95) + 1
    profit_score = np.clip(raw_profit / max_abs, -1, 1).round(4)

    # ── Risk-adjusted return (%) ──────────────────────────────────────────────
    risk_adj_return = (
        (interest_earned - expected_loss)
        / np.maximum(approved["loan_amount"].values, 1) * 100
    ).round(4)

    # ── Customer Lifetime Value ───────────────────────────────────────────────
    # Proxy: income * repayment_quality * product_multiplier
    product_mult = {"Personal Loan": 1.0, "SME Working Capital": 2.5, "BNPL": 0.5}
    pm = approved["loan_type"].map(product_mult).values
    repay_quality = 1 - approved["worst_stage_rank"].values / 4
    income_vals   = [
        cust_map.get(cid, {}).get("monthly_income", 30_000)
        for cid in approved["customer_id"]
    ]
    clv = (
        np.array(income_vals) * repay_quality * pm * 12
        * np.random.uniform(0.8, 1.2, len(approved))
    ).round(2)

    # ── Closure status ────────────────────────────────────────────────────────
    closure_status = np.where(
        writeoff_flag == 1, "Written Off",
        np.where(default_flag == 1, "Delinquent",
                 np.random.choice(["Closed", "Active"], len(approved), p=[0.55, 0.45]))
    )

    df = pd.DataFrame({
        "outcome_id":          [f"OUT{str(i).zfill(8)}" for i in range(1, len(approved) + 1)],
        "loan_id":             approved["loan_id"].values,
        "default_flag":        default_flag.values,
        "writeoff_flag":       writeoff_flag.values,
        "recovery_amount":     recovery_amount,
        "customer_lifetime_value": clv,
        "profitability_score": profit_score,
        "risk_adjusted_return": risk_adj_return,
        "closure_status":      closure_status,
    })

    log.info("Outcome table done. Shape: %s | Default rate: %.1f%% | Write-off rate: %.1f%%",
             df.shape,
             df["default_flag"].mean() * 100,
             df["writeoff_flag"].mean() * 100)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — DATA QUALITY VALIDATION
# ─────────────────────────────────────────────────────────────────────────────

def validate_dataset(customers, loans, repayments, behavioral, outcomes):
    """
    Run a suite of data-quality checks and print a summary report.
    Returns True if all critical checks pass.
    """
    log.info("Running data quality validation …")
    issues  = []
    reports = []

    def check(condition, label, critical=False):
        status = "✅ PASS" if condition else ("❌ FAIL" if critical else "⚠️  WARN")
        reports.append(f"  {status}  {label}")
        if not condition and critical:
            issues.append(label)

    # ── Row counts ────────────────────────────────────────────────────────────
    check(len(customers) >= 25_000,     f"Customers ≥ 25k ({len(customers):,})",       critical=True)
    check(len(loans)     >= 60_000,     f"Loans ≥ 60k ({len(loans):,})",               critical=True)
    check(len(repayments) > 100_000,    f"Repayment records > 100k ({len(repayments):,})")
    check(len(outcomes)  > 0,           f"Outcomes generated ({len(outcomes):,})",      critical=True)

    # ── Primary key uniqueness ────────────────────────────────────────────────
    check(customers["customer_id"].nunique() == len(customers), "customer_id is unique", critical=True)
    check(loans["loan_id"].nunique()         == len(loans),     "loan_id is unique",     critical=True)

    # ── Referential integrity ─────────────────────────────────────────────────
    loan_custs = set(loans["customer_id"])
    all_custs  = set(customers["customer_id"])
    check(loan_custs.issubset(all_custs), "All loan customer_ids exist in customer table", critical=True)

    # ── Default rate in range ─────────────────────────────────────────────────
    dr = outcomes["default_flag"].mean() * 100
    check(8 <= dr <= 22, f"Default rate in 8–22% range ({dr:.2f}%)")

    # ── Missing value rates ───────────────────────────────────────────────────
    for col in ["credit_score", "monthly_income", "employment_type"]:
        miss = customers[col].isna().mean() * 100
        check(miss < 10, f"Missing rate < 10% for customer.{col} ({miss:.2f}%)")

    # ── Credit score range ────────────────────────────────────────────────────
    cs = customers["credit_score"].dropna()
    check(cs.between(300, 900).all(), f"Credit scores in [300, 900]")

    # ── Interest rate range ───────────────────────────────────────────────────
    ir = loans["interest_rate"]
    check(ir.between(8, 36).all(), f"Interest rates in [8, 36]%")

    print("\n" + "═" * 60)
    print("  DATA QUALITY REPORT")
    print("═" * 60)
    for r in reports:
        print(r)
    print("═" * 60)

    if issues:
        print(f"  ⚠️  {len(issues)} critical issue(s) found:")
        for i in issues:
            print(f"     • {i}")
    else:
        print("  All critical checks passed.")
    print("═" * 60 + "\n")

    return len(issues) == 0


# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — STATISTICAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

def print_statistical_summary(customers, loans, outcomes):
    """Print key statistical summaries across all tables."""
    print("\n" + "═" * 60)
    print("  STATISTICAL SUMMARY")
    print("═" * 60)

    print("\n📊 Customer Profile")
    print(customers[["age", "monthly_income", "credit_score",
                      "income_stability_score"]].describe().round(2))

    print("\n📊 Loan Applications")
    print(loans[["loan_amount", "interest_rate",
                 "loan_tenure_months", "emi_amount"]].describe().round(2))

    print("\n📊 Loan Type Distribution")
    print(loans["loan_type"].value_counts())

    print("\n📊 Risk Grade Distribution")
    print(loans["origination_risk_grade"].value_counts().sort_index())

    print("\n📊 Approval Rate by Risk Grade")
    print(loans.groupby("origination_risk_grade")["approval_status"]
          .apply(lambda x: (x == "Approved").mean())
          .round(3))

    print("\n📊 Outcome Summary")
    print(outcomes[["default_flag", "writeoff_flag", "profitability_score",
                     "risk_adjusted_return"]].describe().round(4))

    print("\n📊 Default Rate by Loan Type")
    merged = loans.merge(outcomes[["loan_id", "default_flag"]], on="loan_id", how="inner")
    print(merged.groupby("loan_type")["default_flag"].mean().round(3))
    print("═" * 60 + "\n")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — EXPORT TO CSV
# ─────────────────────────────────────────────────────────────────────────────

def export_to_csv(customers, loans, repayments, behavioral, outcomes,
                  directory: str = OUTPUT_DIR):
    """Export all five tables to CSV files."""
    log.info("Exporting datasets to %s …", directory)
    tables = {
        "01_customer_profile.csv":    customers,
        "02_loan_application.csv":    loans,
        "03_repayment_behavior.csv":  repayments,
        "04_behavioral_signals.csv":  behavioral,
        "05_outcome.csv":             outcomes,
    }
    for fname, df in tables.items():
        path = os.path.join(directory, fname)
        df.to_csv(path, index=False)
        log.info("  Saved %s  (%s rows, %.1f MB)",
                 fname, f"{len(df):,}", os.path.getsize(path) / 1e6)
    log.info("Export complete.")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — ER DIAGRAM (text)
# ─────────────────────────────────────────────────────────────────────────────

ER_DIAGRAM = """
╔══════════════════════════════════════════════════════════╗
║          ENTITY-RELATIONSHIP OVERVIEW                   ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  CUSTOMER_PROFILE ──┐                                    ║
║  (PK: customer_id)  │ 1:N                               ║
║                     ▼                                    ║
║              LOAN_APPLICATION ──────────────────────┐   ║
║              (PK: loan_id,                          │   ║
║               FK: customer_id)                      │   ║
║                     │ 1:N                           │   ║
║                     ▼                               │   ║
║            REPAYMENT_BEHAVIOR              OUTCOME  │   ║
║            (PK: repayment_id,          (PK: outcome_id) ║
║             FK: loan_id)               (FK: loan_id)    ║
║                                                      │   ║
║  BEHAVIORAL_SIGNALS                                  │   ║
║  (PK: signal_id,                                     │   ║
║   FK: customer_id)  ◄────────────────────────────────┘   ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
"""

# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 — PostgreSQL DDL (reference)
# ─────────────────────────────────────────────────────────────────────────────

POSTGRES_DDL = """
-- Run in psql or any PostgreSQL client after \\i this file
-- or paste into DBeaver / pgAdmin

CREATE TABLE customer_profile (
    customer_id              TEXT PRIMARY KEY,
    age                      INT,
    gender                   TEXT,
    city                     TEXT,
    state                    TEXT,
    urban_semiurban_flag     TEXT,
    employment_type          TEXT,
    monthly_income           NUMERIC(12,2),
    income_stability_score   NUMERIC(5,4),
    education_level          TEXT,
    marital_status           TEXT,
    dependents               INT,
    bank_balance_avg         NUMERIC(14,2),
    existing_loans           INT,
    credit_score             INT,
    credit_history_length    INT,
    device_type              TEXT,
    digital_engagement_score INT,
    acquisition_channel      TEXT
);

CREATE TABLE loan_application (
    loan_id                  TEXT PRIMARY KEY,
    customer_id              TEXT REFERENCES customer_profile(customer_id),
    loan_type                TEXT,
    loan_amount              NUMERIC(14,2),
    loan_tenure_months       INT,
    interest_rate            NUMERIC(5,2),
    origination_date         DATE,
    approval_status          TEXT,
    origination_risk_grade   TEXT,
    processing_time_hours    NUMERIC(6,1),
    acquisition_cost         NUMERIC(10,2),
    collateral_flag          INT,
    repayment_frequency      TEXT,
    emi_amount               NUMERIC(12,2)
);

CREATE TABLE repayment_behavior (
    repayment_id             TEXT PRIMARY KEY,
    loan_id                  TEXT REFERENCES loan_application(loan_id),
    due_date                 DATE,
    payment_date             DATE,
    payment_status           TEXT,
    delay_days               INT,
    partial_payment_flag     INT,
    missed_payment_flag      INT,
    bounce_count             INT,
    delinquency_stage        TEXT
);

CREATE TABLE behavioral_signals (
    signal_id                TEXT PRIMARY KEY,
    customer_id              TEXT REFERENCES customer_profile(customer_id),
    month                    DATE,
    spending_volatility      NUMERIC(6,4),
    balance_volatility       NUMERIC(6,4),
    transaction_frequency    INT,
    spending_shock_flag      INT,
    cashflow_consistency_score NUMERIC(6,4),
    app_usage_frequency      INT,
    location_change_frequency INT,
    late_night_transaction_ratio NUMERIC(6,4)
);

CREATE TABLE outcome (
    outcome_id               TEXT PRIMARY KEY,
    loan_id                  TEXT REFERENCES loan_application(loan_id),
    default_flag             INT,
    writeoff_flag            INT,
    recovery_amount          NUMERIC(14,2),
    customer_lifetime_value  NUMERIC(14,2),
    profitability_score      NUMERIC(7,4),
    risk_adjusted_return     NUMERIC(8,4),
    closure_status           TEXT
);

-- Recommended indexes for analytical queries
CREATE INDEX idx_loan_customer   ON loan_application(customer_id);
CREATE INDEX idx_repay_loan      ON repayment_behavior(loan_id);
CREATE INDEX idx_signal_customer ON behavioral_signals(customer_id);
CREATE INDEX idx_outcome_loan    ON outcome(loan_id);
CREATE INDEX idx_loan_orig_date  ON loan_application(origination_date);
CREATE INDEX idx_repay_due_date  ON repayment_behavior(due_date);
"""

# ─────────────────────────────────────────────────────────────────────────────
# CELL 15 — MAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print(ER_DIAGRAM)

    # ── Step 1: Customer profiles ─────────────────────────────────────────────
    customers  = generate_customer_profiles(N_CUSTOMERS)

    # ── Step 2: Loan applications ─────────────────────────────────────────────
    loans      = generate_loan_applications(customers, N_LOANS)

    # ── Step 3: Repayment behavior ────────────────────────────────────────────
    # NOTE: This is the slowest step (~30–90 s depending on hardware)
    repayments = generate_repayment_behavior(loans, customers)

    # ── Step 4: Behavioral signals ────────────────────────────────────────────
    # NOTE: Generates ~60 months × N_active_customers rows — can be large.
    #       Reduce the date_range in generate_behavioral_signals if memory is tight.
    behavioral = generate_behavioral_signals(customers, loans)

    # ── Step 5: Outcomes ──────────────────────────────────────────────────────
    outcomes   = generate_outcomes(loans, repayments, customers)

    # ── Validate ──────────────────────────────────────────────────────────────
    validate_dataset(customers, loans, repayments, behavioral, outcomes)

    # ── Summary ───────────────────────────────────────────────────────────────
    print_statistical_summary(customers, loans, outcomes)

    # ── Export ────────────────────────────────────────────────────────────────
    export_to_csv(customers, loans, repayments, behavioral, outcomes)

    # ── Postgres DDL hint ─────────────────────────────────────────────────────
    ddl_path = os.path.join(OUTPUT_DIR, "create_tables.sql")
    with open(ddl_path, "w") as f:
        f.write(POSTGRES_DDL)
    log.info("PostgreSQL DDL saved to %s", ddl_path)

    print("\n✅  All done!  Files are in the '%s/' folder.\n" % OUTPUT_DIR)
    return customers, loans, repayments, behavioral, outcomes


# ─────────────────────────────────────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    customers, loans, repayments, behavioral, outcomes = main()

# ─────────────────────────────────────────────────────────────────────────────
# CELL 16 — QUICK EXPLORATION (run separately in notebook)
# ─────────────────────────────────────────────────────────────────────────────
# Uncomment any block below in a notebook cell to explore the data.

# # --- Correlation between credit score and default rate
# import matplotlib.pyplot as plt
# merged = loans.merge(outcomes[["loan_id","default_flag"]], on="loan_id")
# merged = merged.merge(customers[["customer_id","credit_score"]], on="customer_id")
# merged["cs_bucket"] = pd.cut(merged["credit_score"], bins=[300,500,600,650,700,750,900])
# dr = merged.groupby("cs_bucket")["default_flag"].mean()
# dr.plot(kind="bar", title="Default Rate by Credit Score Bucket", rot=30)
# plt.ylabel("Default Rate"); plt.tight_layout(); plt.show()

# # --- Delinquency progression heatmap
# stage_counts = repayments.groupby(
#     [pd.to_datetime(repayments["due_date"]).dt.to_period("Q"), "delinquency_stage"]
# ).size().unstack(fill_value=0)
# stage_counts.plot(kind="area", stacked=True,
#                   title="Delinquency Stage Volume by Quarter")
# plt.tight_layout(); plt.show()

# # --- Income vs Approval rate by employment type
# loans_c = loans.merge(customers[["customer_id","employment_type"]], on="customer_id")
# loans_c["approved"] = (loans_c["approval_status"]=="Approved").astype(int)
# print(loans_c.groupby("employment_type")[["approved","loan_amount"]].mean().round(3))



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: C:\Users\abhir\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip
12:06:05 | INFO | Configuration loaded. N_CUSTOMERS=25000  N_LOANS=65000
12:06:05 | INFO | Generating 25000 customer profiles …



╔══════════════════════════════════════════════════════════╗
║          ENTITY-RELATIONSHIP OVERVIEW                   ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  CUSTOMER_PROFILE ──┐                                    ║
║  (PK: customer_id)  │ 1:N                               ║
║                     ▼                                    ║
║              LOAN_APPLICATION ──────────────────────┐   ║
║              (PK: loan_id,                          │   ║
║               FK: customer_id)                      │   ║
║                     │ 1:N                           │   ║
║                     ▼                               │   ║
║            REPAYMENT_BEHAVIOR              OUTCOME  │   ║
║            (PK: repayment_id,          (PK: outcome_id) ║
║             FK: loan_id)               (FK: loan_id)    ║
║                                                      │   ║
║  BEHAVIORAL_SIGNALS            

12:06:06 | INFO | Customer profiles done. Shape: (25000, 19)
12:06:06 | INFO | Generating 65000 loan applications …
12:06:07 | INFO | Loan applications done. Shape: (65000, 14) | Approval rate: 37.8%
12:06:07 | INFO | Generating repayment records (this may take ~30–60 s) …
Repayments: 100%|████████████████████| 24599/24599 [06:51<00:00, 59.79it/s]
12:13:00 | INFO | Repayment table done. Shape: (728384, 10)
12:13:00 | INFO | Generating behavioral signals …
BehSignals: 100%|███████████████████| 23113/23113 [00:25<00:00, 889.26it/s]
12:13:29 | INFO | Behavioral signals done. Shape: (1409893, 11)
12:13:29 | INFO | Generating outcome table …
12:13:29 | INFO | Outcome table done. Shape: (24599, 9) | Default rate: 1.9% | Write-off rate: 1.0%
12:13:29 | INFO | Running data quality validation …
12:13:29 | INFO | Exporting datasets to lending_data …



════════════════════════════════════════════════════════════
  DATA QUALITY REPORT
════════════════════════════════════════════════════════════
  ✅ PASS  Customers ≥ 25k (25,000)
  ✅ PASS  Loans ≥ 60k (65,000)
  ✅ PASS  Repayment records > 100k (728,384)
  ✅ PASS  Outcomes generated (24,599)
  ✅ PASS  customer_id is unique
  ✅ PASS  loan_id is unique
  ✅ PASS  All loan customer_ids exist in customer table
  ⚠️  WARN  Default rate in 8–22% range (1.94%)
  ✅ PASS  Missing rate < 10% for customer.credit_score (0.00%)
  ✅ PASS  Missing rate < 10% for customer.monthly_income (0.00%)
  ✅ PASS  Missing rate < 10% for customer.employment_type (0.00%)
  ✅ PASS  Credit scores in [300, 900]
  ✅ PASS  Interest rates in [8, 36]%
════════════════════════════════════════════════════════════
  All critical checks passed.
════════════════════════════════════════════════════════════


════════════════════════════════════════════════════════════
  STATISTICAL SUMMARY
════════════════════════════════════

12:13:30 | INFO |   Saved 01_customer_profile.csv  (25,000 rows, 3.5 MB)
12:13:30 | INFO |   Saved 02_loan_application.csv  (65,000 rows, 7.2 MB)
12:13:33 | INFO |   Saved 03_repayment_behavior.csv  (728,384 rows, 56.2 MB)
12:13:41 | INFO |   Saved 04_behavioral_signals.csv  (1,409,893 rows, 108.3 MB)
12:13:41 | INFO |   Saved 05_outcome.csv  (24,599 rows, 1.7 MB)
12:13:41 | INFO | Export complete.
12:13:41 | INFO | PostgreSQL DDL saved to lending_data\create_tables.sql



✅  All done!  Files are in the 'lending_data/' folder.

